In [1]:
print("Hello")

Hello


In [5]:
model = "tev1:0.8b"
baseurl= "http://localhost:11434"
apikey = "ollama"

In [9]:
from typesafe_sdk import Choice, Noul, Score, TypeSafeClient

state = {
    "policy": "Returns are allowed within 30 days of purchase.",
    "request": "I bought these headphones 12 days ago and want to send them back.",
}
questions = {
    "eligible": Noul(
        instructions="Is this return within the allowed window?",
    ),
    "category": Choice(
        instructions="What kind of request is this?",
        criteria={
            "return": "The customer wants to send an item back.",
            "exchange": "The customer wants a different item instead.",
            "other": "None of the above.",
        },
    ),
    "sentiment": Score(
        instructions="How positive is the customer's message?",
        criteria=["Negative", "Neutral", "Positive"],
    ),
}

with TypeSafeClient(timeout=120, api_key=apikey, base_url=baseurl) as client:
    result = client.system_one(state=state, questions=questions, model=model )

print(result.nouls["eligible"].noul)          # probability the return is allowed
print(result.choices["category"].choice)      # the chosen category
print(result.scores["sentiment"].score)       # expected level, from 0 to 2

0.8582964448273321
return
1.3579560160932158


In [7]:
result

SystemOneResponse(model='tev1:0.8b', usage=Usage(input_tokens=474, output_tokens=0), answers={'eligible': NoulAnswer(type='noul', noul=0.8582964448273321), 'category': ChoiceAnswer(type='choice', choice='return', confidence=0.9974603656861604, probabilities={'return': 0.9983069104574404, 'exchange': 0.00011094111851726437, 'other': 0.0015821484240424756}), 'sentiment': ScoreAnswer(type='score', score=1.3579560160932158, confidence=0.03693402413982383, legend={0: 'Negative', 1: 'Neutral', 2: 'Positive'}, probabilities={0: 0.192299509907247, 1: 0.25744496409229006, 2: 0.5502555260004629})})

In [12]:
from typesafe_sdk import Choice, Noul, Score, TypeSafeClient

# Simulated comment fetched from Mobitel's official Facebook page post via Graph API
state = {
    "post_context": "Mobitel 4G Unlimited Data Promotion Announcement",
    "comment_text": "Why is my 4G data dropping to 3G speeds in Colombo 03 since yesterday? I paid for unlimited and this is absolute garbage. Fix this or I am switching to Dialog!",
    "user_account_tier": "Postpaid Unlimited",
}

questions = {
    "issue_category": Choice(
        instructions="Classify the core telecom service issue reported by the customer.",
        criteria={
            "data_speed_coverage": "Complaints about slow internet speeds, signal drops, or poor coverage.",
            "billing_charges": "Complaints about unexpected charges, incorrect bills, or data quota deductions.",
            "customer_service": "Complaints about delayed support, unresponsive agents, or store service.",
            "other": "General comments, spam, or issues unrelated to core telecommunication services.",
        },
    ),
    "sentiment_score": Score(
        instructions="Rate the customer frustration level and emotional tone.",
        criteria=["Extremely Negative / Angry", "Neutral / Inquiry", "Positive / Satisfied"],
    ),
    "churn_risk": Noul(
        instructions="What is the probability that this customer will churn or cancel their service based on the threat or severity of dissatisfaction?",
    ),
}

with TypeSafeClient(timeout=120, api_key=apikey, base_url=baseurl) as client:
    result = client.system_one(state=state, questions=questions, model=model )

print(f"Issue Category: {result.choices['issue_category'].choice}")
print(f"Sentiment Level (0-2): {result.scores['sentiment_score'].score}, {round(result.scores['sentiment_score'].score)}")
print(f"Churn Risk Probability: {result.nouls['churn_risk'].noul:.2f}")

# Operational routing based on model outputs
if result.scores['sentiment_score'].score == 0 and result.nouls['churn_risk'].noul > 0.7:
    print("Action: Route immediately to Mobitel Priority Social Media Care Desk for retention workflow.")

Issue Category: data_speed_coverage
Sentiment Level (0-2): 0.04677414145211172, 0
Churn Risk Probability: 0.88


In [1]:
from typesafe_sdk import Choice, Noul, Score, TypeSafeClient
from tqdm.notebook import tqdm

model = "tev1:0.8b"
baseurl= "http://localhost:11434"
apikey = "ollama"

# Simulated comment fetched from Mobitel's official Facebook page post via Graph API
comments_batch = [
    {
        "post_context": "Mobitel 4G Unlimited Data Promotion Announcement",
        "comment_text": "Why is my 4G data dropping to 3G speeds in Colombo 03 since yesterday? I paid for unlimited and this is absolute garbage. Fix this or I am switching to Dialog!",
        "user_account_tier": "Postpaid Unlimited",
    },
    {
        "post_context": "Mobitel 4G Unlimited Data Promotion Announcement",
        "comment_text": "Thank you Mobitel! The new package speeds in Kandy have been super consistent this week.",
        "user_account_tier": "Prepaid",
    },
    {
        "post_context": "Mobitel 4G Unlimited Data Promotion Announcement",
        "comment_text": "Why was I charged an extra 500 LKR on my bill when I have data remaining? This is ridiculous.",
        "user_account_tier": "Postpaid Standard",
    }
]

questions = {
    "issue_category": Choice(
        instructions="Classify the core telecom service issue reported by the customer.",
        criteria={
            "data_speed_coverage": "Complaints about slow internet speeds, signal drops, or poor coverage.",
            "billing_charges": "Complaints about unexpected charges, incorrect bills, or data quota deductions.",
            "customer_service": "Complaints about delayed support, unresponsive agents, or store service.",
            "other": "General comments, spam, or issues unrelated to core telecommunication services.",
        },
    ),
    "sentiment_score": Score(
        instructions="Rate the customer frustration level and emotional tone.",
        criteria=["Extremely Negative / Angry", "Neutral / Inquiry", "Positive / Satisfied"],
    ),
    "churn_risk": Noul(
        instructions="What is the probability that this customer will churn or cancel their service based on the threat or severity of dissatisfaction?",
    ),
}

for comment in tqdm(comments_batch):   
    with TypeSafeClient(timeout=120, api_key=apikey, base_url=baseurl) as client:
        result = client.system_one(state=comment, questions=questions, model=model )

    print(comment)
    print(f"Issue Category: {result.choices['issue_category'].choice}")
    print(f"Sentiment Level (0-2): {result.scores['sentiment_score'].score}, {round(result.scores['sentiment_score'].score)}")
    print(f"Churn Risk Probability: {result.nouls['churn_risk'].noul:.2f}")

    # Operational routing based on model outputs
    if result.scores['sentiment_score'].score == 0 and result.nouls['churn_risk'].noul > 0.7:
        print("Action: Route immediately to Mobitel Priority Social Media Care Desk for retention workflow.")
    
    print("------------------------------------------------------------------------")

  0%|          | 0/3 [00:00<?, ?it/s]

{'post_context': 'Mobitel 4G Unlimited Data Promotion Announcement', 'comment_text': 'Why is my 4G data dropping to 3G speeds in Colombo 03 since yesterday? I paid for unlimited and this is absolute garbage. Fix this or I am switching to Dialog!', 'user_account_tier': 'Postpaid Unlimited'}
Issue Category: data_speed_coverage
Sentiment Level (0-2): 0.04467824952901875, 0
Churn Risk Probability: 0.87
------------------------------------------------------------------------
{'post_context': 'Mobitel 4G Unlimited Data Promotion Announcement', 'comment_text': 'Thank you Mobitel! The new package speeds in Kandy have been super consistent this week.', 'user_account_tier': 'Prepaid'}
Issue Category: data_speed_coverage
Sentiment Level (0-2): 1.3847722008562031, 1
Churn Risk Probability: 0.19
------------------------------------------------------------------------
{'post_context': 'Mobitel 4G Unlimited Data Promotion Announcement', 'comment_text': 'Why was I charged an extra 500 LKR on my bill w

In [6]:
from typesafe_sdk import Choice, Noul, Score, TypeSafeClient
from tqdm.notebook import tqdm

model = "tev1:0.8b"
baseurl = "http://localhost:11434"
apikey = "ollama"

# Simulated comment fetched from Mobitel's official Facebook page post via Graph API
comments_batch = [
    {
        "post_context": "Mobitel 4G Unlimited Data Promotion Announcement",
        "comment_text": "Why is my 4G data dropping to 3G speeds in Colombo 03 since yesterday? I paid for unlimited and this is absolute garbage. Fix this or I am switching to Dialog!",
        "user_account_tier": "Postpaid Unlimited",
    },
    {
        "post_context": "Mobitel 4G Unlimited Data Promotion Announcement",
        "comment_text": "Thank you Mobitel! The new package speeds in Kandy have been super consistent this week.",
        "user_account_tier": "Prepaid",
    },
    {
        "post_context": "Mobitel 4G Unlimited Data Promotion Announcement",
        "comment_text": "Why was I charged an extra 500 LKR on my bill when I have data remaining? This is ridiculous.",
        "user_account_tier": "Postpaid Standard",
    },
    {
        "post_context": "Mobitel 4G Unlimited Data Promotion Announcement",
        "comment_text": "Customer service hotline has been ringing for 45 minutes with no answer. Worst experience ever.",
        "user_account_tier": "Postpaid Standard",
    },
    {
        "post_context": "Mobitel 4G Unlimited Data Promotion Announcement",
        "comment_text": "Can someone DM me the activation code for this package please?",
        "user_account_tier": "Prepaid",
    },
    {
        "post_context": "Mobitel 4G Unlimited Data Promotion Announcement",
        "comment_text": "Signal in Gampaha area completely dropped off the grid today. Unable to work from home.",
        "user_account_tier": "Postpaid Unlimited",
    },
    {
        "post_context": "Mobitel 4G Unlimited Data Promotion Announcement",
        "comment_text": "Amazing network coverage during my trip to Nuwara Eliya. Keep up the good work!",
        "user_account_tier": "Postpaid Standard",
    },
    {
        "post_context": "Mobitel 4G Unlimited Data Promotion Announcement",
        "comment_text": "Charged twice for the same reload through the self care app. I want my money back immediately.",
        "user_account_tier": "Prepaid",
    },
    {
        "post_context": "Mobitel 4G Unlimited Data Promotion Announcement",
        "comment_text": "Is this promotion valid for existing corporate connections or only new users?",
        "user_account_tier": "Corporate",
    },
    {
        "post_context": "Mobitel 4G Unlimited Data Promotion Announcement",
        "comment_text": "Absolute scam. Data quota expired 3 days before the renewal date even though I barely used it.",
        "user_account_tier": "Postpaid Standard",
    },
    {
        "post_context": "Mobitel 4G Unlimited Data Promotion Announcement",
        "comment_text": "Totally satisfied with the speed upgrade in Galle fort area. Cheers Mobitel!",
        "user_account_tier": "Prepaid",
    },
    {
        "post_context": "Mobitel 4G Unlimited Data Promotion Announcement",
        "comment_text": "Your app keeps crashing every time I try to check my remaining balance. Fix your developers work.",
        "user_account_tier": "Postpaid Unlimited",
    },
    {
        "post_context": "Mobitel 4G Unlimited Data Promotion Announcement",
        "comment_text": "Moving to another network tomorrow. Tired of constant disconnection during online lectures in Kurunegala.",
        "user_account_tier": "Prepaid",
    },
    {
        "post_context": "Mobitel 4G Unlimited Data Promotion Announcement",
        "comment_text": "How do I activate international roaming for my upcoming trip to Singapore?",
        "user_account_tier": "Postpaid Standard",
    },
    {
        "post_context": "Mobitel 4G Unlimited Data Promotion Announcement",
        "comment_text": "Connection speed is fine, but pricing is way higher compared to competitors.",
        "user_account_tier": "Postpaid Unlimited",
    },
    {
        "post_context": "Mobitel 4G Unlimited Data Promotion Announcement",
        "comment_text": "Visited the head office branch today and the staff was extremely helpful in resolving my billing dispute.",
        "user_account_tier": "Postpaid Standard",
    },
    {
        "post_context": "Mobitel 4G Unlimited Data Promotion Announcement",
        "comment_text": "Zero signal inside my apartment in Maharagama. Have to step outside just to make a phone call.",
        "user_account_tier": "Prepaid",
    },
    {
        "post_context": "Mobitel 4G Unlimited Data Promotion Announcement",
        "comment_text": "Thanks for the quick response via messenger. Issue resolved!",
        "user_account_tier": "Postpaid Standard",
    },
    {
        "post_context": "Mobitel 4G Unlimited Data Promotion Announcement",
        "comment_text": "Hidden charges on my bill every single month. Filing a complaint with TRCSL if this continues.",
        "user_account_tier": "Postpaid Unlimited",
    },
    {
        "post_context": "Mobitel 4G Unlimited Data Promotion Announcement",
        "comment_text": "Best network for gaming in Jaffna. Low ping and stable connection all day.",
        "user_account_tier": "Prepaid",
    },
    {
        "post_context": "Mobitel 4G Unlimited Data Promotion Announcement",
        "comment_text": "My data pack finished in 2 hours even though the phone was locked. Where did data go?",
        "user_account_tier": "Prepaid",
    },
    {
        "post_context": "Mobitel 4G Unlimited Data Promotion Announcement",
        "comment_text": "Can someone explain why my monthly rental increased without prior notice?",
        "user_account_tier": "Postpaid Standard",
    },
    {
        "post_context": "Mobitel 4G Unlimited Data Promotion Announcement",
        "comment_text": "Rubbish network quality in Matara. Dropped calls every two minutes.",
        "user_account_tier": "Postpaid Unlimited",
    },
    {
        "post_context": "Mobitel 4G Unlimited Data Promotion Announcement",
        "comment_text": "Very happy with the customer care representative who assisted me at the retail outlet.",
        "user_account_tier": "Postpaid Standard",
    },
    {
        "post_context": "Mobitel 4G Unlimited Data Promotion Announcement",
        "comment_text": "Internet stopped working completely right after I recharged. Fix this immediately.",
        "user_account_tier": "Prepaid",
    },
    {
        "post_context": "Mobitel 4G Unlimited Data Promotion Announcement",
        "comment_text": "Is 5G rollout happening in Negombo anytime soon? 4G is getting congested.",
        "user_account_tier": "Postpaid Unlimited",
    },
    {
        "post_context": "Mobitel 4G Unlimited Data Promotion Announcement",
        "comment_text": "Worst telecommunication provider in Sri Lanka. Leaving you guys after 8 years.",
        "user_account_tier": "Postpaid Standard",
    },
    {
        "post_context": "Mobitel 4G Unlimited Data Promotion Announcement",
        "comment_text": "Great value packs introduced recently. Keep it up Mobitel.",
        "user_account_tier": "Prepaid",
    },
    {
        "post_context": "Mobitel 4G Unlimited Data Promotion Announcement",
        "comment_text": "Why is my connection barred when I paid my bill two days before the due date?",
        "user_account_tier": "Postpaid Standard",
    },
    {
        "post_context": "Mobitel 4G Unlimited Data Promotion Announcement",
        "comment_text": "Speeds are extremely slow during peak evening hours in Ratnapura.",
        "user_account_tier": "Postpaid Unlimited",
    },
    {
        "post_context": "Mobitel 4G Unlimited Data Promotion Announcement",
        "comment_text": "Thankful for the free data bonus provided last week.",
        "user_account_tier": "Prepaid",
    },
    {
        "post_context": "Mobitel 4G Unlimited Data Promotion Announcement",
        "comment_text": "Unable to load any streaming apps on mobile data today. Network is completely dead.",
        "user_account_tier": "Postpaid Standard",
    },
    {
        "post_context": "Mobitel 4G Unlimited Data Promotion Announcement",
        "comment_text": "Your automated chatbot is useless. Let me talk to a real human agent.",
        "user_account_tier": "Prepaid",
    },
    {
        "post_context": "Mobitel 4G Unlimited Data Promotion Announcement",
        "comment_text": "Billing errors every month without fail. Tired of calling hotline agents.",
        "user_account_tier": "Postpaid Standard",
    },
    {
        "post_context": "Mobitel 4G Unlimited Data Promotion Announcement",
        "comment_text": "Solid connection throughout my train journey from Colombo to Badulla.",
        "user_account_tier": "Postpaid Unlimited",
    },
    {
        "post_context": "Mobitel 4G Unlimited Data Promotion Announcement",
        "comment_text": "Deactivating my connection tomorrow due to terrible service and dropouts.",
        "user_account_tier": "Postpaid Standard",
    },
    {
        "post_context": "Mobitel 4G Unlimited Data Promotion Announcement",
        "comment_text": "How do I transfer credit to another Mobitel number using the self care app?",
        "user_account_tier": "Prepaid",
    },
    {
        "post_context": "Mobitel 4G Unlimited Data Promotion Announcement",
        "comment_text": "Prices are getting too high for ordinary users. Need more affordable data bundles.",
        "user_account_tier": "Prepaid",
    },
    {
        "post_context": "Mobitel 4G Unlimited Data Promotion Announcement",
        "comment_text": "Excellent network coverage at my university premises.",
        "user_account_tier": "Prepaid",
    },
    {
        "post_context": "Mobitel 4G Unlimited Data Promotion Announcement",
        "comment_text": "Complete waste of money. Data finished within hours of activation.",
        "user_account_tier": "Postpaid Standard",
    },
]

sentiment = [
            "Extremely Negative / Angry",
            "Neutral / Inquiry",
            "Positive / Satisfied",
        ]

questions = {
    "issue_category": Choice(
        instructions="Classify the core telecom service issue reported by the customer.",
        criteria={
            "data_speed_coverage": "Complaints about slow internet speeds, signal drops, or poor coverage.",
            "billing_charges": "Complaints about unexpected charges, incorrect bills, or data quota deductions.",
            "customer_service": "Complaints about delayed support, unresponsive agents, or store service.",
            "other": "General comments, spam, or issues unrelated to core telecommunication services.",
        },
    ),
    "sentiment_score": Score(
        instructions="Rate the customer frustration level and emotional tone.",
        criteria=sentiment,
    ),
    "churn_risk": Noul(
        instructions="What is the probability that this customer will churn or cancel their service based on the threat or severity of dissatisfaction?",
    ),
}

for comment in tqdm(
    comments_batch,
    unit="comment",
    desc="Analyzing Comments",

):
    with TypeSafeClient(timeout=120, api_key=apikey, base_url=baseurl) as client:
        result = client.system_one(state=comment, questions=questions, model=model)

    print(comment)
    print(f"Issue Category: {result.choices['issue_category'].choice}")
    sentiment_score = result.scores['sentiment_score'].score
    print(
        f"Sentiment Level (0-2): {sentiment_score}, {round(sentiment_score)} = {sentiment[round(sentiment_score)]}"
    )
    print(f"Churn Risk Probability: {result.nouls['churn_risk'].noul:.2f}")

    # Operational routing based on model outputs
    if (
        result.scores["sentiment_score"].score == 0
        and result.nouls["churn_risk"].noul > 0.7
    ):
        print(
            "Action: Route immediately to Mobitel Priority Social Media Care Desk for retention workflow."
        )

    print("------------------------------------------------------------------------")

Analyzing Comments:   0%|          | 0/40 [00:00<?, ?comment/s]

{'post_context': 'Mobitel 4G Unlimited Data Promotion Announcement', 'comment_text': 'Why is my 4G data dropping to 3G speeds in Colombo 03 since yesterday? I paid for unlimited and this is absolute garbage. Fix this or I am switching to Dialog!', 'user_account_tier': 'Postpaid Unlimited'}
Issue Category: data_speed_coverage
Sentiment Level (0-2): 0.04333998516305041, 0 = Extremely Negative / Angry
Churn Risk Probability: 0.86
------------------------------------------------------------------------
{'post_context': 'Mobitel 4G Unlimited Data Promotion Announcement', 'comment_text': 'Thank you Mobitel! The new package speeds in Kandy have been super consistent this week.', 'user_account_tier': 'Prepaid'}
Issue Category: data_speed_coverage
Sentiment Level (0-2): 1.3927332598027293, 1 = Neutral / Inquiry
Churn Risk Probability: 0.17
------------------------------------------------------------------------
{'post_context': 'Mobitel 4G Unlimited Data Promotion Announcement', 'comment_text':